# Flickr30K ExpansionNet v2 → MobileNetV3-Large — Hybrid KD: ONNX Schema Preflight

Kiểm tra byte checkpoint, khóa đúng state dict và runtime contract trước khi xuất ONNX.

## Input bắt buộc — đúng 1 input

1. output `MOB_F30K_T1S1_Hybrid_Entry`.

Runtime: `CPU / Accelerator None / Internet Off`.

In [1]:
import csv, hashlib, json, os, shutil
from datetime import datetime, timezone
from pathlib import Path

INPUT_ROOT = Path(os.environ.get("KLCN_INPUT_ROOT", "/kaggle/input"))

def now(): return datetime.now(timezone.utc).isoformat()
def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for block in iter(lambda: handle.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()
def read_json(path): return json.loads(Path(path).read_text(encoding="utf-8"))
def write_json(path, payload):
    Path(path).write_text(json.dumps(payload, ensure_ascii=False, indent=2) + "\n", encoding="utf-8")
def unique_named(name):
    matches = sorted(path for path in INPUT_ROOT.rglob(name) if path.is_file())
    assert len(matches) == 1, (name, [str(x) for x in matches])
    return matches[0]
def nested(payload, dotted):
    value = payload
    for key in dotted.split("|"):
        value = value[key]
    return value

import torch

CFG = json.loads('{"short": "F30K_T1S1_Hybrid", "title": "Flickr30K ExpansionNet v2 → MobileNetV3-Large — Hybrid KD", "dataset": "Flickr30K", "method": "Hybrid Sequence and Dense-Mapped Logit Knowledge Distillation", "teacher": "ExpansionNet v2", "student": "MobileNetV3-Large", "seed": 2026, "result_name": "F30K_T1S1_Hybrid_2026_RESULT.json", "aggregate_name": "F30K_T1S1_Hybrid_Aggregate_RESULT.json", "checkpoint_name": "best.pt", "checkpoint_hash_source": "artifacts|best.pt|sha256", "baseline": 0.4769577031590234, "kd": 0.3825670867861916, "expected_keys": ["encoder_state_dict", "decoder_state_dict"], "vocab_name": "f30k_vocabulary_minfreq5.json", "vocab_array_key": "tokens", "vocab_size": 7456, "inputs": ["output `F30K_T1S1_Hybrid_2026`", "output `F30K_T1S1_Hybrid_Aggregate`", "output `F30K_Data`"]}')
EXPERIMENT_ID = "MOB_" + CFG["short"] + "_ONNX_Preflight"
OUT = Path(os.environ.get("KLCN_OUTPUT_ROOT", f"/kaggle/working/{EXPERIMENT_ID}"))
OUT.mkdir(parents=True, exist_ok=True)

entry_name = f"MOB_{CFG['short']}_ENTRY_LOCK.json"
entry_path = unique_named(entry_name)
entry = read_json(entry_path)
assert entry["status"] == "locked"
assert entry["selected_seed"] == CFG["seed"]
assert entry["test_set_used"] is False

checkpoint_ref = entry["checkpoint"]
checkpoint_path = entry_path.parent / checkpoint_ref["relative_path"]
assert checkpoint_path.is_file()
assert checkpoint_path.stat().st_size == checkpoint_ref["bytes"]
assert sha256(checkpoint_path) == checkpoint_ref["sha256"]

payload = torch.load(checkpoint_path, map_location="cpu", weights_only=False)
missing = [key for key in CFG["expected_keys"] if key not in payload]
assert not missing, ("Missing checkpoint keys", missing, sorted(payload.keys()))

state_summaries = {}
for key in CFG["expected_keys"]:
    state = payload[key]
    tensors = {name: value for name, value in state.items() if torch.is_tensor(value)}
    assert tensors, key
    state_summaries[key] = {
        "tensor_count": len(tensors),
        "parameter_count": int(sum(value.numel() for value in tensors.values())),
        "floating_dtypes": sorted({str(value.dtype) for value in tensors.values() if value.is_floating_point()}),
        "first_keys": sorted(tensors)[:12],
    }

contract = {
    "schema_version": "mobile_onnx_preflight_v1",
    "created_at_utc": now(),
    "status": "passed",
    "experiment_id": EXPERIMENT_ID,
    "model_id": CFG["short"],
    "selected_seed": CFG["seed"],
    "teacher": CFG["teacher"],
    "student": CFG["student"],
    "dataset": CFG["dataset"],
    "method": CFG["method"],
    "checkpoint_sha256": sha256(checkpoint_path),
    "required_state_dict_keys": CFG["expected_keys"],
    "state_dict_summary": state_summaries,
    "fixed_runtime_contract": {
        "image": [1, 3, 224, 224],
        "visual_memory": [1, 49, 256],
        "decoder_steps": 31,
        "beam_size": 3,
        "vocabulary_size": CFG["vocab_size"],
    },
    "authorized_next": [
        f"MOB_{CFG['short']}_ONNX_FP32",
        f"MOB_{CFG['short']}_Calib_Lock",
    ],
    "parallel_next": True,
    "selection_split": "validation",
    "test_set_status": "sealed",
    "test_set_used": False,
}
contract_path = OUT / f"MOB_{CFG['short']}_ONNX_PREFLIGHT.json"
write_json(contract_path, contract)
print("ONNX schema preflight   : PASSED")
print("Model                   :", CFG["short"])
print("Checkpoint states       :", CFG["expected_keys"])
for key, summary in state_summaries.items():
    print(key, "tensors/parameters:", summary["tensor_count"], "/", f"{summary['parameter_count']:,}")
print("Next in parallel        : ONNX FP32 + Calibration Lock")
print("Test                    : SEALED / NOT READ")

ONNX schema preflight   : PASSED
Model                   : F30K_T1S1_Hybrid
Checkpoint states       : ['encoder_state_dict', 'decoder_state_dict']
encoder_state_dict tensors/parameters: 313 / 3,255,470
decoder_state_dict tensors/parameters: 59 / 5,084,960
Next in parallel        : ONNX FP32 + Calibration Lock
Test                    : SEALED / NOT READ
